<a href="https://colab.research.google.com/github/abyanrizz/practicallinearalgebra/blob/main/Chapter_13_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#chapter 13

Eigendecomposition is a pearl of linear algebra. What is a pearl? Let me quote directly
from the book 20,000 Leagues Under the Sea:
For poets, a pearl is a tear from the sea; for Orientals, it’s a drop of solidified dew; for
the ladies it’s a jewel they can wear on their fingers, necks, and ears that’s oblong in
shape, glassy in luster, and formed from mother-of-pearl; for chemists, it’s a mixture
of calcium phosphate and calcium carbonate with a little gelatin protein; and finally,

for naturalists, it’s a simple festering secretion from the organ that produces mother-of-
pearl in certain bivalves.

—Jules Verne
The point is that the same object can be seen in different ways depending on its use.
So it is with eigendecomposition: eigendecomposition has a geometric interpretation
(axes of rotational invariance), a statistical interpetation (directions of maximal cova‐
riance), a dynamical-systems interpretation (stable system states), a graph-theoretic
interpretation (the impact of a node on its network), a financial-market interpreta‐
tion (identifying stocks that covary), and many more.
Eigendecomposition (and the SVD, which, as you’ll learn in the next chapter, is
closely related to eigendecomposition) is among the most important contributions of
linear algebra to data science. The purpose of this chapter is to provide you an intu‐
itive understanding of eigenvalues and eigenvectors—the results of eigendecomposi‐
tion of a matrix. Along the way, you’ll learn about diagonalization and more special
properties of symmetric matrices. After extending eigendecomposition to the SVD in
Chapter 14, you’ll see a few applications of eigendecomposition in Chapter 15.

213

1 Approximately −.6 and 1.6.

Just for Squares

Eigendecomposition is defined only for square matrices. It is not possible to eigende‐
compose an M × N matrix unless M = N. Nonsquare matrices can be decomposed
using the SVD. Every square matrix of size M × M has M eigenvalues (scalars) and M
corresponding eigenvectors. The purpose of eigendecomposition is to reveal those M
vector-scalar pairs.
Interpretations of Eigenvalues and Eigenvectors
There are several ways of interpreting eigenvalues/vectors that I will describe in
the next sections. Of course, the math is the same regardless, but having mutliple
perspectives can facilitate intuition, which in turn will help you understand how and
why eigendecomposition is important in data science.
Geometry
I’ve actually already introduced you to the geometric concept of eigenvectors in
Chapter 5. In Figure 5-5, we discovered that there was a special combination of a
matrix and a vector such that the matrix stretched—but did not rotate—that vector.
That vector is an eigenvector of the matrix, and the amount of stretching is the
eigenvalue.
Figure 13-1 shows vectors before and after multiplication by a 2 × 2 matrix. The
two vectors in the left plot (v1 and v2) are eigenvectors whereas the two vectors in
the right plot are not. The eigenvectors point in the same direction before and after
postmultiplying the matrix. The eigenvalues encode the amount of stretching; tryup and down together), or whether there are independent subcategories within that
space (meaning that some coins or groups of coins change in value independently of
the value of other coins).
We can test this hypothesis by performing a principal components analysis on a
dataset that contains the prices of various cryptocoins over time. If the entire cryp‐
tomarket operates as a single entity, then the scree plot (a graph of the eigenvalues
of the dataset’s covariance matrix) would reveal that one component accounts for
most of the variance of the system, and all other components account for very little
variance (graph A in Figure 13-2). In contrast, if the cryptomarket had, say, three
major subcategories with independent price movements, then we would expect to see
three large eigenvalues (graph B in Figure 13-2).

Figure 13-2. Simulated scree plots of multivariate datasets (data is simulated to illustrate
outcome possibilities)
Noise Reduction
Most datasets contain noise. Noise refers to variance in a dataset that is either unex‐
plained (e.g., random variation) or unwanted (e.g., electrical line noise artifacts in
radio signals). There are many ways to attenuate or eliminate noise, and the optimal
noise-reduction strategy depends on the nature and origin of the noise and on the
characteristics of the signal.
One method of reducing random noise is to identify the eigenvalues and eigenvectors
of a system, and “project out” directions in the data space associated with small eigen‐
values. The assumption is that random noise makes a relatively small contribution to
the total variance. “Projecting out” a data dimension means to reconstruct a dataset
after setting some eigenvalues that are below some threshold to zero.
You’ll see an example of using eigendecomposition to reduce noise in Chapter 15.
216 | Chapter 13: Eigendecomposition

Dimension Reduction (Data Compression)
Information communications technologies like phones, internet, and TV create and
transmit a huge amount of data, such as pictures and videos. Transmitting data can
be time-consuming and expensive, and it is beneficial to compress the data before
transmitting it. Compression means to reduce the size of the data (in terms of bytes)
while having minimal impact on the quality of the data. For example, a TIFF format
image file might be 10 MB, while the JPG converted version might be .1 MB while
still retaining reasonably good quality.
One way to dimension-reduce a dataset is to take its eigendecomposition, drop
the eigenvalues and eigenvectors associated with small directions in the data space,
and then transmit only the relatively larger eigenvector/value pairs. It is actually
more common to use the SVD for data compression (and you’ll see an example in
Chapter 15), although the principle is the same.
Modern data compression algorithms are actually faster and more efficient than the
method previously described, but the idea is the same: decompose the dataset into
a set of basis vectors that capture the most important features of the data, and then
reconstruct a high-quality version of the original data.
Finding Eigenvalues
To eigendecompose a square matrix, you first find the eigenvalues, and then use each
eigenvalue to find its corresponding eigenvector. The eigenvalues are like keys that
you insert into the matrix to unlock the mystical eigenvector.
Finding the eigenvalues of a matrix is super easy in Python:
matrix = np.array([
[1,2],
[3,4]
])
# get the eigenvalues
evals = np.linalg.eig(matrix)[0]
The two eigenvalues (rounded to the nearest hundredth) are −0.37 and 5.37.
But the important question isn’t which function returns the eigenvalues; instead, the
important question is how are the eigenvalues of a matrix identified?

Finding Eigenvalues | 217

3 As I wrote in Chapter 5, Python will return a result, but that is the scalar broadcast subtracted, which is not a
linear algebra operation.
To find the eigenvalues of a matrix, we start with the eigenvalue equation shown in
Equation 13-1 and do some simple arithemetic, as shown in Equation 13-2.
Equation 13-2. Eigenvalue equation, reorganized
Av = λv
Av − λv = 0
A − λI v = 0
The first equation is an exact repeat of the eigenvalue equation. In the second
equation, we simply subtracted the right-hand side to set the equation to the zeros
vector.
The transition from the second to the third equation requires some explanation. The
left-hand side of the second equation has two vector terms, both of which involve v.
So we factor out the vector. But that leaves us with the subtraction of a matrix and a
scalar (A − λ), which is not a defined operation in linear algebra.3

So instead, we shift

the matrix by λ. That brings us to the third equation. (Side note: the expression λI is
sometimes called a scalar matrix.)
What does that third equation mean? It means that the eigenvector is in the null space
of the matrix shifted by its eigenvalue.
If it helps you understand the concept of the eigenvector as the null-space vector of
the shifted matrix, you can think of adding two additional equations:
A = A − λI
Av = 0
Why is that statement so insightful? Remember that we ignore trivial solutions in
linear algebra, so we do not consider v = 0 to be an eigenvector. And that means that
the matrix shifted by its eigenvalue is singular, because only singular matrices have a
nontrivial null space.
And what else do we know about singular matrices? We know that their determinant
is zero. Hence:
A − λI = 0

218 | Chapter 13: Eigendecomposition

Believe it or not, that’s the key to finding eigenvalues: shift the matrix by the
unknown eigenvalue λ, set its determinant to zero, and solve for λ. Let’s see how
this looks for a 2 × 2 matrix:
a b
c d
− λ
1 0
0 1
= 0
a − λ b
c d − λ
= 0
a − λ d − λ − bc = 0
λ
2
− a + d λ + ad − bc = 0
You could apply the quadratic formula to solve for the two λ values. But the answer
itself isn’t important; the important thing is to see the logical progression of mathe‐
matical concepts established earlier in this book:
• The matrix-vector multiplication acts like scalar-vector multiplication (the eigen‐
value equation).
• We set the eigenvalue equation to the zeros vector, and factor out common terms.
• This reveals that the eigenvector is in the null space of the matrix shifted by
the eigenvalue. We do not consider the zeros vector to be an eigenvector, which
means the shifted matrix is singular.
• Therefore, we set the determinant of the shifted matrix to zero and solve for the
unknown eigenvalue.
The determinant of an eigenvalue-shifted matrix set to zero is called the characteristic
polynomial of the matrix.
Notice that in the previous example, we started with a 2 × 2 matrix and got a λ
2
term,
which means this is a second-order polynomial equation. You might remember from
your high-school algebra class that an nth order polynomial has n solutions, some of
which might be complex-valued (this is called the fundamental theorem of algebra).
So, there will be two values of λ that can satisfy the equation.
The matching 2s are no coincidence: the characteristic polynomial of an M × M
matrix will have a λM

term. That is the reason why an M × M matrix will have M

eigenvalues.

Finding Eigenvalues | 219

Tedious Practice Problems
At this point in a traditional linear algebra textbook, you would
be tasked with finding the eigenvalues of dozens of 2 × 2 and
3 × 3 matrices by hand. I have mixed feelings about these kinds of
exercises: on the one hand, solving problems by hand really helps
internalize the mechanics of finding eigenvalues; but on the other
hand, I want to focus this book on concepts, code, and applications,
without getting bogged down by tedious arithmetic. If you feel
inspired to solve eigenvalue problems by hand, then go for it! You
can find myriad such problems in traditional textbooks or online.
But I took the bold (and perhaps controversial) decision to avoid
hand-solved problems in this book, and instead to have exercises
that focus on coding and comprehension.

Finding Eigenvectors
As with eigenvalues, finding eigenvectors is super-duper easy in Python:

In [ ]:
evals,evecs = np.linalg.eig(matrix)
print(evals), print(evecs)
[-0.37228132 5.37228132]
[[-0.82456484 -0.41597356]

The eigenvectors are in the columns of the matrix evecs and are in the same order as
the eigenvalues (that is, the eigenvector in the first column of matrix evecs is paired
with the first eigenvalue in vector evals). I like to use the variable names evals and
evecs, because they are short and meaningful. You might also see people use variable
names L and V or D and V. The L is for Λ (the capital of λ) and the V is for V,
the matrix in which each column i is eigenvector vi

. The D is for diagonal, because
eigenvalues are often stored in a diagonal matrix, for reasons I will explain later in
this chapter.

Eigenvectors in the Columns, Not the Rows!
The most important thing to keep in mind about eigenvectors
when coding is that they are stored in the columns of the matrix,
not the rows! Such dimensional-indexing errors are easy to make
with square matrices (because you might not get Python errors),
but accidentally using the rows instead of the columns of the eigen‐
vectors matrix can have disastrous consequences in applications.
When in doubt, remember the discussion from Chapter 2 that
common convention in linear algebra is to assume that vectors are
in column orientation.

220 | Chapter 13: Eigendecomposition

OK, but again, the previous code shows how to get a NumPy function to return
the eigenvectors of a matrix. You could have learned that from the np.linalg.eig
docstring. The important question is Where do eigenvectors come from, and how do we
find them?
Actually, I’ve already written how to find eigenvectors: find the vector v that is in the
null space of the matrix shifted by λ. In other words:
vi ∈ N A − λi
I

Let’s see a numerical example. Follwing is a matrix and its eigenvalues:
1 2
2 1

λ1 = 3, λ2 = −1

Let’s focus on the first eigenvalue. To reveal its eigenvector, we shift the matrix by 3
and find a vector in its null space:
1 − 3 2
2 1 − 3
=
−2 2
2 −2

−2 2
2 −2
1
1
=
0
0

This means that [1 1] is an eigenvector of the matrix associated with an eigenvalue of 3.
I found that null space vector just by looking at the matrix. How are the null space
vectors (that is, the eigenvectors of the matrix) actually identified in practice?
Null space vectors can be found by using Gauss-Jordan to solve a system of equations,
where the coefficients matrix is the λ-shifted matrix and the constants vector is the
zeros vector. That’s a good way to conceptualize the solution. In implementation,
more stable numerical methods are applied for finding eigenvalues and eigenvectors,
including QR decomposition and a procedure called the power method.
Sign and Scale Indeterminacy of Eigenvectors
Let me return to the numerical example in the previous section. I wrote that [1 1] was
an eigenvector of the matrix because that vector is a basis for the null space of the
matrix shifted by its eigenvalue of 3.
Look back at the shifted matrix and ask yourself, is [1 1] the only possible basis vector
for the null space? Not even close! You could also use [4 4] or [−5.4 −5.4] or...I
think you see where this is going: any scaled version of vector [1 1] is a basis for that
null space. In other words, if v is an eigenvector of a matrix, then so is αv for any
real-valued α except zero.

In [ ]:
evals,evecs = np.linalg.eig(matrix)
D = np.diag(evals)

By the way, it’s often interesting and insightful in mathematics to rearrange equations
by solving for different variables. Consider the following list of equivalent declara‐
tions:
AV = VΛ
A = VΛV−1
Λ = V
−1AV
The second equation shows that matrix A becomes diagonal inside the space of V
(that is, V moves us into the “diagonal space,” and then V

−1 gets us back out of the
diagonal space). This can be interpreted in the context of basis vectors: the matrix A
is dense in the standard basis, but then we apply a set of transformations (V) to rotate
the matrix into a new set of basis vectors (the eigenvectors) in which the information
is sparse and represented by a diagonal matrix. (At the end of the equation, we need
to get back into the standard basis space, hence the V
−1.)

Diagonalizing a Square Matrix | 223

The Special Awesomeness of Symmetric Matrices
You already know from earlier chapters that symmetric matrices have special proper‐
ties that make them great to work with. Now you are ready to learn two more special
properties that relate to eigendecomposition.
Orthogonal Eigenvectors
Symmetric matrices have orthogonal eigenvectors. That means that all eigenvectors of
a symmetric matrix are pair-wise orthogonal. Let me start with an example, then I’ll
discuss the implications of eigenvector orthogonality, and finally I’ll show the proof:

In [ ]:
# just some random symmetric matrix
A = np.random.randint(-3,4,(3,3))
A = A.T@A
# its eigendecomposition
L,V = np.linalg.eig(A)
# all pairwise dot products
print( np.dot(V[:,0],V[:,1]) )
print( np.dot(V[:,0],V[:,2]) )
print( np.dot(V[:,1],V[:,2]) )